Análisis exploratorio de la brecha salarial de Género en México con datos de la ENIGH de 2010 a 2025 por promedio anual. 

Expoloración de datos

In [2]:
# Exploración de datos
import pandas as pd
import logging

# Configurar logging
logging.basicConfig(
    filename= 'limpieza_datos.log',
    level= logging.INFO,
    format= '%(asctime)s - %(levelname)s - %(message)s'
)

df_genero = pd.read_csv("data/salario_genero.csv")

df_genero.columns
df_genero.shape
df_genero.dtypes

Nation ID           str
Nation              str
Quarter ID        int64
Quarter             str
Sex ID            int64
Sex                 str
Monthly Wage    float64
Workforce         int64
Time              int64
dtype: object

In [3]:
# Exploración de datos
df_genero.head()
df_genero.info()
df_genero.describe()


<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Nation ID     120 non-null    str    
 1   Nation        120 non-null    str    
 2   Quarter ID    120 non-null    int64  
 3   Quarter       120 non-null    str    
 4   Sex ID        120 non-null    int64  
 5   Sex           120 non-null    str    
 6   Monthly Wage  120 non-null    float64
 7   Workforce     120 non-null    int64  
 8   Time          120 non-null    int64  
dtypes: float64(1), int64(4), str(4)
memory usage: 8.6 KB


,Quarter ID,Sex ID,Monthly Wage,Workforce,Time
count,120.000000,120.000000,120.000000,1.200000e+02,1.200000e+02
mean,20173.316667,1.500000,4204.895991,2.637467e+07,1.500140e+12
std,44.405879,0.502096,975.366776,6.328439e+06,1.401940e+11
min,20101.000000,1.000000,2940.271308,1.707132e+07,1.265004e+12
25%,20133.750000,1.000000,3326.547858,1.989874e+07,1.381298e+12
50%,20172.500000,1.500000,4049.861505,2.658538e+07,1.497589e+12
75%,20212.250000,2.000000,4702.387299,3.224473e+07,1.621832e+12
max,20251.000000,2.000000,6854.283894,3.826931e+07,1.738390e+12


In [5]:
df_genero["Quarter"].unique()

<StringArray>
['2010-Q1', '2010-Q2', '2010-Q3', '2010-Q4', '2011-Q1', '2011-Q2', '2011-Q3',
 '2011-Q4', '2012-Q1', '2012-Q2', '2012-Q3', '2012-Q4', '2013-Q1', '2013-Q2',
 '2013-Q3', '2013-Q4', '2014-Q1', '2014-Q2', '2014-Q3', '2014-Q4', '2015-Q1',
 '2015-Q2', '2015-Q3', '2015-Q4', '2016-Q1', '2016-Q2', '2016-Q3', '2016-Q4',
 '2017-Q1', '2017-Q2', '2017-Q3', '2017-Q4', '2018-Q1', '2018-Q2', '2018-Q3',
 '2018-Q4', '2019-Q1', '2019-Q2', '2019-Q3', '2019-Q4', '2020-Q1', '2020-Q3',
 '2020-Q4', '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1', '2022-Q2',
 '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1',
 '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1']
Length: 60, dtype: str

In [6]:
df_genero.groupby("Sex")["Monthly Wage"].describe()

,count,mean,std,min,25%,50%,75%,max
Sex,,,,,,,,
Hombre,60.0,4716.575604,914.103454,3814.950625,4013.575563,4272.760893,5306.883162,6854.283894
Mujer,60.0,3693.216378,742.100270,2940.271308,3128.991485,3314.922490,4242.903643,5391.139285


In [7]:
df_genero.groupby(["Time","Sex"])["Monthly Wage"].mean()

Time           Sex   
1265004000000  Hombre    4048.124001
               Mujer     3037.057223
1272690000000  Hombre    3979.107922
               Mujer     3052.803053
1280638800000  Hombre    3901.177221
                            ...     
1722492000000  Mujer     5391.139285
1730440800000  Hombre    6607.577274
               Mujer     5217.921972
1738389600000  Hombre    6573.454827
               Mujer     5158.636867
Name: Monthly Wage, Length: 120, dtype: float64

Limpieza de datos

In [8]:
#función de limpieza

def limpiar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')

    )
    return df

#aplicar función de limpieza
df_genero = limpiar_columnas(df_genero)
df_genero.columns

Index(['nation_id', 'nation', 'quarter_id', 'quarter', 'sex_id', 'sex',
       'monthly_wage', 'workforce', 'time'],
      dtype='str')

In [10]:
# valores nulos
nulos = df_genero.isnull().sum()

logging.info(f"Valores nulos:\n{nulos}")

In [ ]:
# Cambiar nombre a dataframe + tabla dinámica. 

brecha = df_genero.pivot_table(
    index = "time",
    columns = "sex",
    values = "monthly_wage"
)

brecha.head()


sex,Hombre,Mujer
time,,
1265004000000,4048.124001,3037.057223
1272690000000,3979.107922,3052.803053
1280638800000,3901.177221,3054.264227
1288591200000,3870.284954,3036.822241
1296540000000,3993.445997,3116.343110


In [ ]:
# Cálculo brecha %
brecha["Brecha_%"] = (brecha["Hombre"] - brecha["Mujer"]) / brecha["Hombre"] * 100

brecha = brecha.reset_index()

# Transformar tiempo a año
brecha["Fecha"] = pd.to_datetime(brecha["time"], unit="ms")
brecha["Año"] = brecha["Fecha"].dt.year

brecha = brecha.set_index("Año")

brecha[["Hombre", "Mujer", "Brecha_%"]].head()

sex,Hombre,Mujer,Brecha_%
Año,,,
2010,4048.124001,3037.057223,24.976181
2010,3979.107922,3052.803053,23.279210
2010,3901.177221,3054.264227,21.709165
2010,3870.284954,3036.822241,21.534919
2011,3993.445997,3116.343110,21.963559


In [17]:
# Variable para promediar brecha x año
salario_anual = brecha.groupby("Año")[["Hombre", "Mujer"]].mean()
salario_anual.head()

sex,Hombre,Mujer
Año,,
2010,3949.673524,3045.236686
2011,3907.589175,3032.752060
2012,3999.155867,3110.602348
2013,4012.887315,3135.506329
2014,3956.457364,3080.500623
